In [1]:
import os
import requests
from tqdm import tqdm
import pandas as pd

import project_dirs as pdir

In [2]:
import random
import numpy as np

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

set_seed(seed=8)

In [3]:
# Desired local directory
csv_dir = os.path.join(pdir.CSV_DIR, "CLWD.csv")

# List of files to download
csv_Data = pd.read_csv(csv_dir)

WSI_names = csv_Data["WSI_ID"].to_list()
print(f"Total WSIs: {len(WSI_names)}")
WSI_labels = csv_Data["Benchmark_Label_7class"].to_list()
Patient_IDs = csv_Data["SampleNumber"].to_list()

Patients = sorted(set(Patient_IDs))
print(f"Total Patients: {len(Patients)}")

Total WSIs: 408
Total Patients: 209


In [8]:
WSI_ages = csv_Data["Age"].to_list()
min_age = np.min(WSI_ages)
max_age = np.max(WSI_ages)

print(f"Min Age: {min_age}, Max Age: {max_age}")

normalized_ages = [(age - min_age) / (max_age - min_age) for age in WSI_ages]

Min Age: 24, Max Age: 80


In [6]:
def find_matching_indices(strings, query):
    """
    Return the indices where query exactly matches an element in strings.

    Parameters
    ----------
    strings : list[str]
        List of strings to search.
    query : str
        String to search for.

    Returns
    -------
    list[int]
        Indices of matching strings.
    """
    return [i for i, s in enumerate(strings) if s == query]

In [4]:
def find_smallest_wsi_index(indices, wsis):
    """
    Select WSIs at the given indices, find the WSI with the
    smallest numerical identifier, and return its original index.

    Parameters
    ----------
    indices : list[int]
        Indices to select from the WSI list.
    wsis : list[str]
        List of WSI identifiers, e.g. ['WSI-11', 'WSI-13', 'WSI-14', 'WSI-9'].

    Returns
    -------
    int
        Original index of the WSI with the smallest numerical identifier.
    """
    selected = [(i, wsis[i]) for i in indices]

    smallest_index, _ = min(
        selected,
        key=lambda x: int(x[1].split("-")[-1])
    )

    return smallest_index

In [5]:
def select_dataframe_rows(df, new_indices):
    """
    Create a new DataFrame containing rows from df at the specified
    positional indices.

    Parameters
    ----------
    df : pandas.DataFrame
        Original DataFrame.
    new_indices : list[int]
        Positional indices of the rows to select.

    Returns
    -------
    pandas.DataFrame
        New DataFrame containing the selected rows.
    """
    return df.iloc[new_indices].copy()

In [6]:
def merge_indices(patient_indices, indices):
    """
    Merge indices from a dictionary and a list, removing duplicates.

    Parameters
    ----------
    patient_indices : dict
        Dictionary where values are lists of indices.
    indices : list
        List of additional indices.

    Returns
    -------
    list
        Merged list of unique indices.
    """

    # Flatten all indices from the dictionary
    dict_indices = [
        index
        for index_list in patient_indices.values()
        for index in index_list
    ]

    # Merge and remove duplicates
    merged_indices = list(set(dict_indices + indices))

    return merged_indices

In [7]:
def find_patients_with_different_labels(df):
    """
    Find patients who have multiple WSIs with different WSI labels.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing:
        - SampleNumber: patient ID
        - Benchmark_Label_7class: WSI label
        - WSI_ID: WSI name

    Returns
    -------
    patient_numbers : list
        Patient IDs with multiple slides and different labels.

    patient_indices : dict
        Dictionary mapping each patient ID to the DataFrame indices
        corresponding to their WSIs.
    """

    # Find patients with more than one unique WSI label
    patients = (
        df.groupby("SampleNumber")["Benchmark_Label_7class"]
        .nunique()
    )

    # Keep patients with different labels
    patient_numbers = patients[patients > 1].index.tolist()

    # Get DataFrame indices for each patient
    patient_indices = {
        patient: df.index[df["SampleNumber"] == patient].tolist()
        for patient in patient_numbers
    }

    return patient_numbers, patient_indices

## Patients with multiple Slide Labels
patient_numbers, patient_indices = find_patients_with_different_labels(csv_Data)

In [8]:

new_df_indices = []

for P in Patients:
    print(f"Patient ID: {P}")
    idxs = find_matching_indices(Patient_IDs, P)
    print(f"No. of Slides: {len(idxs)}")

    if len(idxs) == 1:
        new_df_indices.append(idxs[0])
    else:
        new_df_indices.append(find_smallest_wsi_index(idxs, WSI_names))

new_df_indices = merge_indices(patient_indices, new_df_indices)

Patient ID: 8007623
No. of Slides: 1
Patient ID: 8008524
No. of Slides: 1
Patient ID: 8013888
No. of Slides: 3
Patient ID: 8017082
No. of Slides: 2
Patient ID: 8020981
No. of Slides: 1
Patient ID: 8022410
No. of Slides: 2
Patient ID: 8029758
No. of Slides: 2
Patient ID: 8030821
No. of Slides: 2
Patient ID: 8060308
No. of Slides: 4
Patient ID: 8103477
No. of Slides: 1
Patient ID: 8111526
No. of Slides: 3
Patient ID: 8126701
No. of Slides: 1
Patient ID: 8133249
No. of Slides: 2
Patient ID: 8143088
No. of Slides: 1
Patient ID: 8146413
No. of Slides: 2
Patient ID: 8163836
No. of Slides: 2
Patient ID: 8170462
No. of Slides: 1
Patient ID: 8180155
No. of Slides: 3
Patient ID: 8189481
No. of Slides: 2
Patient ID: 8203346
No. of Slides: 2
Patient ID: 8215472
No. of Slides: 2
Patient ID: 8219606
No. of Slides: 1
Patient ID: 8221926
No. of Slides: 1
Patient ID: 8221946
No. of Slides: 1
Patient ID: 8223039
No. of Slides: 1
Patient ID: 8224566
No. of Slides: 1
Patient ID: 8225155
No. of Slides: 1
P

In [9]:
new_df = select_dataframe_rows(csv_Data, new_df_indices)

In [11]:
new_df.to_csv(os.path.join(pdir.CSV_DIR, 'CLWD_OneSlide.csv'), index=False)

In [13]:
new_csv_data = pd.read_csv(os.path.join(pdir.CSV_DIR, 'CLWD_OneSlide.csv'))

WSI_names = new_csv_data["WSI_ID"].to_list()
print(f"Total WSIs: {len(WSI_names)}")
Patient_IDs = new_csv_data["SampleNumber"].to_list()

Patients = sorted(set(Patient_IDs))
print(f"Total Patients: {len(Patients)}")

Total WSIs: 215
Total Patients: 209
